In [1]:
import os
import sys
import time
import numpy as np
import pandas as pd
import requests
from pathlib import Path
import json
import plotly.express as px


## Setup

In [18]:
# -----------------------------
# Imports
# -----------------------------
from pathlib import Path
import requests
import pandas as pd
import numpy as np


# -----------------------------
# Paths / settings
# -----------------------------
OUTPUT_DIR = Path(r"C:\SDSU_SCIL_CensusData\data\processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FINAL_PARQUET = OUTPUT_DIR / "acs_county_year_dataset_fixed.parquet"

START_YEAR = 2011
END_YEAR = 2024

# Use your key if already defined. Otherwise this falls back to None.
try:
    CENSUS_API_KEY
except NameError:
    CENSUS_API_KEY = None


# -----------------------------
# Variables to download
# -----------------------------
# Detail variables are used for stable direct estimates.
# Education percentages use profile variables because B15003 is not available
# for all older ACS years, including 2011.

ACS_DETAIL_VARIABLES = {
    # Demographics
    "B01003_001E": "total_population",
    "B01002_001E": "median_age",

    # Economy / income
    "B19013_001E": "median_household_income",

    # Housing
    "B25077_001E": "median_home_value",
    "B25064_001E": "median_gross_rent",

    # Poverty
    "B17001_001E": "poverty_universe",
    "B17001_002E": "poverty_count",

    # Labor force
    "B23025_001E": "population_16_plus",
    "B23025_002E": "labor_force",
    "B23025_003E": "civilian_labor_force",
    "B23025_004E": "employed",
    "B23025_005E": "unemployed",
}

ACS_PROFILE_EDUCATION_VARIABLES = {
    "DP02_0068PE": "less_than_high_school_pct",
    "DP02_0069PE": "high_school_grad_pct",
    "DP02_0071PE": "bachelors_or_higher_pct",
}


MISSING_SENTINELS = [
    -666666666,
    -999999999,
    -888888888,
    -222222222,
    -333333333,
    -555555555,
]


# -----------------------------
# Cleaning helpers
# -----------------------------

def clean_numeric_series(s):
    """
    Convert ACS values to numeric and replace Census missing/suppressed codes with NaN.
    """
    s = pd.to_numeric(s, errors="coerce")
    s = s.replace(MISSING_SENTINELS, np.nan)
    s = s.mask(s < -1000000, np.nan)
    return s


def clean_rate_column(df, col):
    """
    Keep percentage/rate columns between 0 and 100.
    """
    if col in df.columns:
        df.loc[(df[col] < 0) | (df[col] > 100), col] = np.nan


def clean_positive_column(df, col, min_valid=None):
    """
    Removes negative and tiny placeholder-like values from positive quantity columns.
    """
    if col not in df.columns:
        return

    df.loc[df[col] < 0, col] = np.nan

    if min_valid is not None:
        df.loc[df[col] < min_valid, col] = np.nan


# -----------------------------
# Census download functions
# -----------------------------

def download_census_counties(year, dataset, variables, api_key=None):
    """
    Download one ACS 5-year county-level table from the Census API.

    dataset:
        "" or None -> /acs/acs5 detail endpoint
        "profile" -> /acs/acs5/profile endpoint
    """
    base_url = f"https://api.census.gov/data/{year}/acs/acs5"
    dataset = (dataset or "").strip("/")
    url = f"{base_url}/{dataset}" if dataset else base_url

    params = {
        "get": ",".join(["NAME"] + list(variables.keys())),
        "for": "county:*",
        "in": "state:*",
    }

    if api_key:
        params["key"] = api_key

    response = requests.get(url, params=params, timeout=60)

    if response.status_code != 200:
        raise RuntimeError(
            f"Request failed for {year} {dataset}: {response.status_code}\n{response.text}"
        )

    rows = response.json()
    df = pd.DataFrame(rows[1:], columns=rows[0])

    df["year"] = int(year)
    df["GEOID"] = (
        df["state"].astype(str).str.zfill(2)
        + df["county"].astype(str).str.zfill(3)
    )

    df = df.rename(columns=variables)

    for col in variables.values():
        df[col] = clean_numeric_series(df[col])

    return df


def download_acs_detail_counties(year, variables, api_key=None):
    """
    ACS detail-table variables use the base /acs/acs5 endpoint.
    """
    return download_census_counties(
        year=year,
        dataset="",
        variables=variables,
        api_key=api_key,
    )


def download_acs_profile_counties(year, variables, api_key=None):
    """
    ACS profile variables use the /acs/acs5/profile endpoint.
    """
    return download_census_counties(
        year=year,
        dataset="profile",
        variables=variables,
        api_key=api_key,
    )


# -----------------------------
# Build one county-year dataframe
# -----------------------------

def build_county_year_dataset(year, api_key=None):
    """
    Download ACS county data, compute clean indicators,
    and return one wide county-year dataframe.
    """
    detail = download_acs_detail_counties(
        year=year,
        variables=ACS_DETAIL_VARIABLES,
        api_key=api_key,
    )

    education = download_acs_profile_counties(
        year=year,
        variables=ACS_PROFILE_EDUCATION_VARIABLES,
        api_key=api_key,
    )

    keep_education = ["GEOID", "year"] + list(ACS_PROFILE_EDUCATION_VARIABLES.values())

    df = detail.merge(
        education[keep_education],
        on=["GEOID", "year"],
        how="left",
    )

    # -----------------------------
    # Poverty
    # -----------------------------
    df["poverty_rate"] = np.where(
        df["poverty_universe"] > 0,
        df["poverty_count"] / df["poverty_universe"] * 100,
        np.nan,
    )

    # -----------------------------
    # Labor indicators
    # -----------------------------
    df["employment_population_ratio"] = np.where(
        df["population_16_plus"] > 0,
        df["employed"] / df["population_16_plus"] * 100,
        np.nan,
    )

    df["labor_force_participation_rate"] = np.where(
        df["population_16_plus"] > 0,
        df["labor_force"] / df["population_16_plus"] * 100,
        np.nan,
    )

    df["employment_rate"] = np.where(
        df["civilian_labor_force"] > 0,
        df["employed"] / df["civilian_labor_force"] * 100,
        np.nan,
    )

    df["unemployment_rate"] = np.where(
        df["civilian_labor_force"] > 0,
        df["unemployed"] / df["civilian_labor_force"] * 100,
        np.nan,
    )

    # -----------------------------
    # Clean impossible rate values
    # -----------------------------
    rate_cols = [
        "poverty_rate",
        "less_than_high_school_pct",
        "high_school_grad_pct",
        "bachelors_or_higher_pct",
        "employment_population_ratio",
        "labor_force_participation_rate",
        "employment_rate",
        "unemployment_rate",
    ]

    for col in rate_cols:
        clean_rate_column(df, col)

    # -----------------------------
    # Clean impossible quantity / dollar values
    # -----------------------------
    clean_positive_column(df, "total_population", min_valid=1)
    clean_positive_column(df, "median_age", min_valid=1)
    clean_positive_column(df, "median_household_income", min_valid=1000)
    clean_positive_column(df, "median_home_value", min_valid=10000)
    clean_positive_column(df, "median_gross_rent", min_valid=100)
    clean_positive_column(df, "population_16_plus", min_valid=1)
    clean_positive_column(df, "labor_force", min_valid=0)
    clean_positive_column(df, "civilian_labor_force", min_valid=0)
    clean_positive_column(df, "employed", min_valid=0)
    clean_positive_column(df, "unemployed", min_valid=0)

    # Extra sanity for median age
    df.loc[(df["median_age"] < 0) | (df["median_age"] > 120), "median_age"] = np.nan

    # -----------------------------
    # Keep final dashboard columns
    # -----------------------------
    keep_cols = [
        "NAME",
        "total_population",
        "median_age",
        "median_household_income",
        "poverty_rate",
        "less_than_high_school_pct",
        "high_school_grad_pct",
        "bachelors_or_higher_pct",
        "median_home_value",
        "median_gross_rent",
        "state",
        "county",
        "year",
        "GEOID",
        "population_16_plus",
        "labor_force",
        "civilian_labor_force",
        "employed",
        "unemployed",
        "employment_population_ratio",
        "labor_force_participation_rate",
        "employment_rate",
        "unemployment_rate",
    ]

    keep_cols = [c for c in keep_cols if c in df.columns]

    return df[keep_cols]


# -----------------------------
# Build all years and save
# -----------------------------

def build_all_county_years(
    start_year=START_YEAR,
    end_year=END_YEAR,
    output_dir=OUTPUT_DIR,
    final_parquet=FINAL_PARQUET,
    api_key=None,
    overwrite_yearly=False,
):
    """
    Build yearly county datasets and save one final parquet.

    Saves yearly CSVs and the final parquet to:
        C:\\SDSU_SCIL_CensusData\\data\\processed
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    all_years = []

    for year in range(start_year, end_year + 1):
        yearly_file = output_dir / f"acs_county_year_{year}.csv"

        if yearly_file.exists() and not overwrite_yearly:
            print(f"Loading existing yearly file: {yearly_file}")
            df_year = pd.read_csv(
                yearly_file,
                dtype={
                    "state": str,
                    "county": str,
                    "GEOID": str,
                },
            )
            df_year["GEOID"] = df_year["GEOID"].astype(str).str.zfill(5)
        else:
            print(f"Downloading ACS 5-year county data for {year}...")
            df_year = build_county_year_dataset(
                year=year,
                api_key=api_key,
            )

            df_year.to_csv(yearly_file, index=False)
            print("Saved yearly CSV:", yearly_file)

        all_years.append(df_year)

    df_all = pd.concat(all_years, ignore_index=True)

    df_all["GEOID"] = df_all["GEOID"].astype(str).str.zfill(5)
    df_all["year"] = df_all["year"].astype(int)

    df_all.to_parquet(final_parquet, index=False)

    print("\nSaved final parquet:", final_parquet)
    print("Rows:", len(df_all))
    print("Columns:", len(df_all.columns))

    return df_all


# -----------------------------
# Sanity checks
# -----------------------------

def run_sanity_checks(df):
    """
    Print basic sanity checks after building the dataset.
    """
    check_cols = [
        "total_population",
        "median_age",
        "median_household_income",
        "median_home_value",
        "median_gross_rent",
        "poverty_rate",
        "less_than_high_school_pct",
        "high_school_grad_pct",
        "bachelors_or_higher_pct",
        "employment_rate",
        "unemployment_rate",
    ]

    check_cols = [c for c in check_cols if c in df.columns]

    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())

    print("\nSanity describe:")
    display(df[check_cols].describe())

    print("\nSample rows:")
    sample_cols = [
        "NAME",
        "year",
        "total_population",
        "median_age",
        "median_household_income",
        "median_home_value",
        "median_gross_rent",
        "poverty_rate",
        "employment_rate",
    ]
    sample_cols = [c for c in sample_cols if c in df.columns]

    display(df[sample_cols].head(20))


# -----------------------------
# Run build
# -----------------------------
# Keep overwrite_yearly=True the first time so it replaces the old bad yearly CSVs.

county_wide = build_all_county_years(
    start_year=START_YEAR,
    end_year=END_YEAR,
    output_dir=OUTPUT_DIR,
    final_parquet=FINAL_PARQUET,
    api_key=CENSUS_API_KEY,
    overwrite_yearly=True,
)

run_sanity_checks(county_wide)

Saved yearly CSV: C:\SDSU_SCIL_CensusData\data\processed\acs_county_year_2011.csv
Saved yearly CSV: C:\SDSU_SCIL_CensusData\data\processed\acs_county_year_2012.csv
Saved yearly CSV: C:\SDSU_SCIL_CensusData\data\processed\acs_county_year_2013.csv
Saved yearly CSV: C:\SDSU_SCIL_CensusData\data\processed\acs_county_year_2014.csv
Saved yearly CSV: C:\SDSU_SCIL_CensusData\data\processed\acs_county_year_2015.csv
Saved yearly CSV: C:\SDSU_SCIL_CensusData\data\processed\acs_county_year_2016.csv
Saved yearly CSV: C:\SDSU_SCIL_CensusData\data\processed\acs_county_year_2017.csv
Saved yearly CSV: C:\SDSU_SCIL_CensusData\data\processed\acs_county_year_2018.csv
Saved yearly CSV: C:\SDSU_SCIL_CensusData\data\processed\acs_county_year_2019.csv
Saved yearly CSV: C:\SDSU_SCIL_CensusData\data\processed\acs_county_year_2020.csv
Saved yearly CSV: C:\SDSU_SCIL_CensusData\data\processed\acs_county_year_2021.csv
Saved yearly CSV: C:\SDSU_SCIL_CensusData\data\processed\acs_county_year_2022.csv
Saved yearly CSV

,total_population,median_age,median_household_income,median_home_value,median_gross_rent,poverty_rate,less_than_high_school_pct,high_school_grad_pct,bachelors_or_higher_pct,employment_rate,unemployment_rate
count,4.509100e+04,45091.000000,45083.000000,4.505400e+04,45035.000000,45090.000000,18875.000000,25152.000000,22007.000000,45090.000000,45090.000000
mean,1.008663e+05,41.108217,52310.113768,1.564100e+05,764.380482,16.297054,23.351915,10.076260,15.684578,93.218050,6.781950
std,3.225429e+05,5.304864,16697.219222,1.034201e+05,253.137277,8.128038,10.094034,3.069078,4.555079,3.949534,3.949534
min,3.300000e+01,21.400000,10499.000000,1.680000e+04,243.000000,0.000000,0.000000,0.000000,3.100000,59.106802,0.000000
25%,1.117900e+04,38.000000,41468.500000,9.440000e+04,602.000000,11.056738,16.300000,8.300000,12.500000,91.460378,4.179995
50%,2.598700e+04,41.000000,50033.000000,1.272000e+05,706.000000,14.824552,20.900000,9.800000,15.200000,94.002559,5.997441
75%,6.685850e+04,44.100000,60546.500000,1.801000e+05,857.000000,19.391723,27.800000,11.500000,18.400000,95.820005,8.539622
max,1.010572e+07,86.200000,181765.000000,1.633900e+06,2922.000000,67.073712,86.000000,91.000000,98.000000,100.000000,40.893198



Sample rows:


,NAME,year,total_population,median_age,median_household_income,median_home_value,median_gross_rent,poverty_rate,employment_rate
0,"Clay County, North Carolina",2011,10506.0,49.2,36711.0,162100.0,584.0,21.791908,87.969245
1,"Cumberland County, North Carolina",2011,316478.0,30.9,44861.0,123400.0,820.0,16.611103,88.180265
2,"Guilford County, North Carolina",2011,483081.0,36.3,46288.0,156200.0,729.0,16.218744,89.944446
3,"Jackson County, North Carolina",2011,39574.0,36.4,36826.0,165800.0,618.0,19.524934,92.988184
4,"Pasquotank County, North Carolina",2011,40511.0,36.0,45298.0,174000.0,763.0,19.851941,87.715464
5,"Robeson County, North Carolina",2011,133033.0,34.1,30874.0,68900.0,572.0,30.630150,89.175602
6,"Swain County, North Carolina",2011,13932.0,40.9,40719.0,119700.0,630.0,22.452858,90.628149
7,"Wake County, North Carolina",2011,879658.0,34.3,65289.0,227600.0,869.0,10.131818,93.086605
8,"Anson County, North Carolina",2011,26788.0,39.7,34659.0,81600.0,640.0,21.600296,85.092382
9,"McDowell County, North Carolina",2011,44825.0,41.5,35230.0,100700.0,533.0,18.502776,89.104931


In [19]:
print("Unique states:", county_wide["state"].nunique())
print("Rows by year:")
print(county_wide.groupby("year").size())

print("First 20 states:")
print(county_wide["NAME"].str.split(",").str[-1].str.strip().head(20).tolist())

Unique states: 52
Rows by year:
year
2011    3221
2012    3221
2013    3221
2014    3220
2015    3220
2016    3220
2017    3220
2018    3220
2019    3220
2020    3221
2021    3221
2022    3222
2023    3222
2024    3222
dtype: int64
First 20 states:
['North Carolina', 'North Carolina', 'North Carolina', 'North Carolina', 'North Carolina', 'North Carolina', 'North Carolina', 'North Carolina', 'North Carolina', 'North Carolina', 'North Carolina', 'North Carolina', 'North Carolina', 'North Carolina', 'North Carolina', 'North Carolina', 'North Carolina', 'North Carolina', 'North Carolina', 'North Carolina']


In [2]:
from pathlib import Path
import pandas as pd

# --------------------------------------------------
# File paths
# --------------------------------------------------
ACS_PARQUET = Path(
    r"C:\SDSU_SCIL_CensusData\data\processed"
    r"\acs_county_year_dataset_fixed.parquet"
)

FEMA_CSV = Path(
    r"C:\SDSU_SCIL_CensusData\data\raw"
    r"\NRI_Table_Counties\NRI_Table_Counties.csv"
)

OUTPUT_PARQUET = Path(
    r"C:\SDSU_SCIL_CensusData\data\processed"
    r"\acs_county_year_fema_risk.parquet"
)


# --------------------------------------------------
# Load datasets
# --------------------------------------------------
acs = pd.read_parquet(ACS_PARQUET)

fema_columns = [
    # County identifiers
    "STCOFIPS",
    "STATE",
    "STATEABBRV",
    "COUNTY",
    "NRI_VER",

    # Overall FEMA risk
    "RISK_VALUE",
    "RISK_SCORE",
    "RISK_RATNG",
    "RISK_SPCTL",

    # Expected annual loss
    "EAL_SCORE",
    "EAL_RATNG",
    "EAL_SPCTL",
    "EAL_VALT",

    # Social vulnerability
    "SOVI_SCORE",
    "SOVI_RATNG",
    "SOVI_SPCTL",

    # Community resilience
    "RESL_SCORE",
    "RESL_RATNG",
    "RESL_SPCTL",
    "RESL_VALUE",
]

fema = pd.read_csv(
    FEMA_CSV,
    usecols=fema_columns,
    dtype={"STCOFIPS": str},
    low_memory=False,
)


# --------------------------------------------------
# Standardize county identifiers
# --------------------------------------------------
acs["GEOID"] = (
    acs["GEOID"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(5)
)

fema["GEOID"] = (
    fema["STCOFIPS"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(5)
)


# --------------------------------------------------
# Rename FEMA columns for clarity
# --------------------------------------------------
fema = fema.rename(
    columns={
        "STATE": "fema_state",
        "STATEABBRV": "fema_state_abbr",
        "COUNTY": "fema_county",
        "NRI_VER": "fema_nri_version",

        "RISK_VALUE": "fema_risk_value",
        "RISK_SCORE": "fema_risk_score",
        "RISK_RATNG": "fema_risk_rating",
        "RISK_SPCTL": "fema_risk_percentile",

        "EAL_SCORE": "fema_eal_score",
        "EAL_RATNG": "fema_eal_rating",
        "EAL_SPCTL": "fema_eal_percentile",
        "EAL_VALT": "fema_expected_annual_loss",

        "SOVI_SCORE": "fema_social_vulnerability_score",
        "SOVI_RATNG": "fema_social_vulnerability_rating",
        "SOVI_SPCTL": "fema_social_vulnerability_percentile",

        "RESL_SCORE": "fema_resilience_score",
        "RESL_RATNG": "fema_resilience_rating",
        "RESL_SPCTL": "fema_resilience_percentile",
        "RESL_VALUE": "fema_resilience_value",
    }
)

fema = fema.drop(columns="STCOFIPS")


# FEMA should contain one row per county
if fema["GEOID"].duplicated().any():
    duplicate_geoids = fema.loc[
        fema["GEOID"].duplicated(keep=False),
        "GEOID"
    ].unique()

    raise ValueError(
        "Duplicate FEMA county GEOIDs found: "
        f"{duplicate_geoids[:20].tolist()}"
    )


# --------------------------------------------------
# Merge FEMA data onto every ACS county-year row
# --------------------------------------------------
combined = acs.merge(
    fema,
    on="GEOID",
    how="left",
    validate="many_to_one",
    indicator=True,
)


# --------------------------------------------------
# Merge diagnostics
# --------------------------------------------------
print("ACS rows before merge:", len(acs))
print("Rows after merge:", len(combined))

print("\nMerge results:")
print(combined["_merge"].value_counts(dropna=False))

unmatched = (
    combined.loc[combined["_merge"] == "left_only", ["GEOID", "NAME"]]
    .drop_duplicates()
    .sort_values("GEOID")
)

print("\nUnmatched ACS counties:", len(unmatched))

if not unmatched.empty:
    print(unmatched.head(25).to_string(index=False))


# Remove diagnostic column after inspection
combined = combined.drop(columns="_merge")


# --------------------------------------------------
# Save combined parquet
# --------------------------------------------------
OUTPUT_PARQUET.parent.mkdir(parents=True, exist_ok=True)

combined.to_parquet(
    OUTPUT_PARQUET,
    index=False
)

print("\nSaved combined parquet:")
print(OUTPUT_PARQUET)

print("\nFinal shape:", combined.shape)

print("\nFEMA columns added:")
print(
    [
        column
        for column in combined.columns
        if column.startswith("fema_")
    ]
)

ACS rows before merge: 45091
Rows after merge: 45091

Merge results:
_merge
both          44983
left_only       108
right_only        0
Name: count, dtype: int64

Unmatched ACS counties: 12
GEOID                               NAME
02261 Valdez-Cordova Census Area, Alaska
02270   Wade Hampton Census Area, Alaska
09001      Fairfield County, Connecticut
09003       Hartford County, Connecticut
09005     Litchfield County, Connecticut
09007      Middlesex County, Connecticut
09009      New Haven County, Connecticut
09011     New London County, Connecticut
09013        Tolland County, Connecticut
09015        Windham County, Connecticut
46113       Shannon County, South Dakota
51515             Bedford city, Virginia

Saved combined parquet:
C:\SDSU_SCIL_CensusData\data\processed\acs_county_year_fema_risk.parquet

Final shape: (45091, 42)

FEMA columns added:
['fema_state', 'fema_state_abbr', 'fema_county', 'fema_risk_value', 'fema_risk_score', 'fema_risk_rating', 'fema_risk_percentile', '